# Data Preprocessing and Feature Engineering
### Adult Dataset Assignment

## Step 1: Data Exploration and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('adult_with_headers__1_.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Summary Statistics
print('Summary Statistics:')
df.describe()

In [ ]:
# Data Types
print('Data Types:')
print(df.dtypes)

In [ ]:
# Handle Missing Values
# Replace '?' with NaN
df.replace(' ?', np.nan, inplace=True)
df.replace('?', np.nan, inplace=True)

print('Missing Values before imputation:')
print(df.isnull().sum())

# Impute missing values with mode (most frequent value)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print('\nMissing Values after imputation:')
print(df.isnull().sum())

## Step 2: Scaling Techniques

In [ ]:
numerical_cols = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

# Standard Scaling
scaler_std = StandardScaler()
df_std = df.copy()
df_std[numerical_cols] = scaler_std.fit_transform(df[numerical_cols])

print('Standard Scaled values (first 3 rows):')
print(df_std[numerical_cols].head(3))

In [ ]:
# Min-Max Scaling
scaler_mm = MinMaxScaler()
df_mm = df.copy()
df_mm[numerical_cols] = scaler_mm.fit_transform(df[numerical_cols])

print('Min-Max Scaled values (first 3 rows):')
print(df_mm[numerical_cols].head(3))

### Discussion: When to use each scaling technique?

**Standard Scaling (Z-score):**
- Use when data follows a normal distribution
- Preferred for algorithms like Logistic Regression, SVM, PCA
- Works well when there are outliers since it uses mean and std deviation
- Output range is not bounded

**Min-Max Scaling:**
- Use when you need values between a fixed range (0 to 1)
- Preferred for Neural Networks and image processing
- Sensitive to outliers since it uses min and max values
- Best when data does not follow normal distribution

## Step 3: Encoding Techniques

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print('Unique values per categorical column:')
for col in categorical_cols:
    print(f'{col}: {df[col].nunique()} unique values')

In [ ]:
# One-Hot Encoding for columns with less than 5 categories
ohe_cols = [col for col in categorical_cols if df[col].nunique() < 5]
label_cols = [col for col in categorical_cols if df[col].nunique() >= 5]

print('One-Hot Encoding applied to:', ohe_cols)
print('Label Encoding applied to:', label_cols)

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=ohe_cols)

# Apply Label Encoding
le = LabelEncoder()
for col in label_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print('\nEncoded DataFrame shape:', df_encoded.shape)
df_encoded.head()

### Discussion: Pros and Cons of Encoding Techniques

**One-Hot Encoding:**
- Pros: No ordinal relationship assumed between categories, works well with algorithms like Logistic Regression
- Cons: Creates many new columns (curse of dimensionality), not suitable for high cardinality columns

**Label Encoding:**
- Pros: Simple, doesn't increase number of columns, memory efficient
- Cons: Assigns numbers that imply ordinal relationship which may mislead the model

## Step 4: Feature Engineering

In [ ]:
# New Feature 1: capital_net (capital_gain - capital_loss)
# Rationale: Net capital change is more meaningful than separate gain/loss columns
df_encoded['capital_net'] = df['capital_gain'] - df['capital_loss']
print('Feature 1 - capital_net sample:')
print(df_encoded['capital_net'].describe())

In [ ]:
# New Feature 2: age_group
# Rationale: Grouping age into meaningful life stages for better pattern detection
df_encoded['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 45, 65, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)
df_encoded['age_group'] = le.fit_transform(df_encoded['age_group'].astype(str))
print('Feature 2 - age_group sample:')
print(df_encoded['age_group'].value_counts())

In [ ]:
# Log Transformation on capital_gain (highly skewed feature)
# Rationale: capital_gain is heavily right-skewed with many zeros and few large values
# log1p handles zero values gracefully
df_encoded['capital_gain_log'] = np.log1p(df['capital_gain'])

print('Original capital_gain skewness:', df['capital_gain'].skew())
print('Log transformed skewness:', df_encoded['capital_gain_log'].skew())

## Step 5: Isolation Forest (Outlier Detection)

In [ ]:
# Apply Isolation Forest to detect and remove outliers
iso = IsolationForest(contamination=0.05, random_state=42)
df_encoded['anomaly'] = iso.fit_predict(df_encoded[numerical_cols])

print('Anomaly detection results:')
print(df_encoded['anomaly'].value_counts())
print('\n1 = Normal, -1 = Outlier')

# Remove outliers
df_clean = df_encoded[df_encoded['anomaly'] == 1].drop('anomaly', axis=1)
print('\nShape before outlier removal:', df_encoded.shape)
print('Shape after outlier removal:', df_clean.shape)

## Final Preprocessed Dataset

In [ ]:
print('Final dataset shape:', df_clean.shape)
print('\nFinal columns:')
print(df_clean.columns.tolist())
df_clean.head()